# Chapter 7: Functions as First-Class Objects
*From: Fluent Python by Luciano Ramalho*

---

Python functions are **first-class objects**: they can be created at runtime, assigned to variables, stored in data structures, passed as arguments to other functions, and returned as the results of other functions. This is a cornerstone that enables functional programming patterns, higher-order functions, and the entire decorator ecosystem.

## TL;DR

- **First-class functions** mean functions are objects: assign them, pass them, return them, inspect them.
- **Higher-order functions** take or return functions -- `sorted(key=...)`, `map()`, `filter()`.
- **List comprehensions** and **generator expressions** replace most uses of `map()` and `filter()` with clearer syntax.
- **lambda** creates small anonymous functions limited to a single expression -- best used as callbacks.
- Python has **nine callable types**; use `callable()` to check.
- Implementing `__call__` makes any class instance callable like a function.
- **Keyword-only** (`*`) and **positional-only** (`/`) parameters give precise control over call signatures.
- The **operator** module (`itemgetter`, `attrgetter`, `mul`) and **functools.partial** reduce trivial lambdas.

---
## 1. First-Class Functions

A "first-class object" is a program entity that can be:
- Created at runtime
- Assigned to a variable or stored in a data structure
- Passed as an argument to a function
- Returned as the result of a function

Functions in Python satisfy all four criteria.

In [ ]:
# Functions are objects: create, assign, inspect
def factorial(n):
    """Returns n!"""
    return 1 if n < 2 else n * factorial(n - 1)

# Assign to a variable
fact = factorial
print(f"fact(5) = {fact(5)}")
print(f"type: {type(factorial)}")
print(f"__doc__: {factorial.__doc__}")
print(f"__name__: {factorial.__name__}")

In [ ]:
# Pass a function as an argument
print(list(map(factorial, range(8))))

# Store functions in a data structure
operations = {"square": lambda x: x**2, "double": lambda x: x*2}
print(operations["square"](7))

---
## 2. Higher-Order Functions

A **higher-order function** takes a function as an argument or returns one as its result.

Classic examples: `sorted()`, `map()`, `filter()`, `functools.reduce()`.

In [ ]:
# sorted() with a key function
fruits = ["strawberry", "fig", "apple", "cherry", "raspberry", "banana"]

# Sort by length
print("By length:", sorted(fruits, key=len))

# Sort by reversed spelling (rhyme order)
def reverse(word):
    return word[::-1]

print("Rhyme order:", sorted(fruits, key=reverse))

In [ ]:
# min/max with key functions
print("Shortest:", min(fruits, key=len))
print("Last alphabetically:", max(fruits))

---
## 3. Modern Replacements for map, filter, and reduce

List comprehensions and generator expressions replace most uses of `map()` and `filter()` -- and they are more readable.

In [ ]:
# map + filter vs list comprehension
# Old style
old = list(map(factorial, filter(lambda n: n % 2, range(6))))

# Modern style -- clearer!
new = [factorial(n) for n in range(6) if n % 2]

print(f"map/filter: {old}")
print(f"listcomp:   {new}")
assert old == new

In [ ]:
# reduce vs sum
from functools import reduce
from operator import add

print("reduce:", reduce(add, range(100)))
print("sum:   ", sum(range(100)))  # much clearer!

# all() and any() are also reducing built-ins
print("all([1, 2, 3]):", all([1, 2, 3]))
print("any([0, 0, 1]):", any([0, 0, 1]))

---
## 4. Anonymous Functions (lambda)

`lambda` creates an anonymous function limited to a **single expression**. Best used as a throwaway callback in higher-order function calls.

In [ ]:
# lambda for a quick sort key
fruits = ["strawberry", "fig", "apple", "cherry", "raspberry", "banana"]
print(sorted(fruits, key=lambda word: word[::-1]))

# If a lambda is hard to read, refactor to a named function
def clean(s):
    return s.lower().strip()

print(clean("  Hello World  "))

---
## 5. The Nine Flavors of Callable Objects

Python has nine callable types. Use `callable()` to test any object:

| # | Callable type | Example |
|---|---|---|
| 1 | User-defined functions | `def`, `lambda` |
| 2 | Built-in functions | `len`, `abs` |
| 3 | Built-in methods | `dict.get` |
| 4 | Methods | Functions defined in a class body |
| 5 | Classes | Calling a class invokes `__new__` + `__init__` |
| 6 | Class instances | If the class defines `__call__` |
| 7 | Generator functions | Functions using `yield` |
| 8 | Native coroutines | `async def` functions |
| 9 | Async generators | `async def` with `yield` |

In [ ]:
# callable() in action
print(callable(abs))        # built-in function
print(callable(str))        # class (calling it creates instances)
print(callable("hello"))    # string is NOT callable
print(callable(lambda: 0))  # lambda is callable
print(callable([].append))  # method is callable

---
## 6. User-Defined Callable Types (`__call__`)

Any class implementing `__call__` makes its instances callable. This is ideal for function-like objects that need **persistent internal state**.

In [ ]:
import random

class BingoCage:
    """Picks items from a shuffled list without repeats."""
    def __init__(self, items):
        self._items = list(items)
        random.shuffle(self._items)

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError("pick from empty BingoCage")

    def __call__(self):
        return self.pick()  # shortcut: bingo() == bingo.pick()

bingo = BingoCage(range(5))
print(f"bingo.pick() = {bingo.pick()}")
print(f"bingo()      = {bingo()}")   # __call__ in action
print(f"callable?    {callable(bingo)}")

---
## 7. Positional-Only and Keyword-Only Parameters

Python 3.8+ gives precise control over how callers pass arguments:
- Parameters **before** `/` are **positional-only**
- Parameters **after** `*` are **keyword-only**

In [ ]:
# Keyword-only parameters: after *
def tag(name, *content, class_=None, **attrs):
    """Generate one or more HTML tags."""
    if class_ is not None:
        attrs["class"] = class_
    attr_pairs = (f' {a}="{v}"' for a, v in sorted(attrs.items()))
    attr_str = "".join(attr_pairs)
    if content:
        elements = (f"<{name}{attr_str}>{c}</{name}>" for c in content)
        return "\n".join(elements)
    else:
        return f"<{name}{attr_str} />"

print(tag("br"))
print(tag("p", "hello", "world", class_="sidebar"))
print(tag("img", src="photo.jpg", alt="A photo"))

In [ ]:
# Positional-only parameters: before / (Python 3.8+)
def divmod_safe(a, b, /):
    """a and b must be passed positionally."""
    return (a // b, a % b)

print(divmod_safe(10, 3))

# Combining / and * in one signature
def combined(pos_only, /, normal, *, kw_only):
    return f"{pos_only}, {normal}, {kw_only}"

print(combined(1, 2, kw_only=3))
print(combined(1, normal=2, kw_only=3))

---
## 8. The operator Module and functools.partial

The **operator** module provides function equivalents for operators, replacing trivial lambdas. **functools.partial** freezes some arguments of a callable.

In [ ]:
from functools import reduce
from operator import mul, itemgetter, attrgetter

# operator.mul replaces lambda a, b: a * b
def factorial_op(n):
    return reduce(mul, range(1, n + 1))

print(f"10! = {factorial_op(10)}")

# itemgetter: sort tuples by a specific field
metro_data = [
    ("Tokyo", "JP", 36.933),
    ("Delhi NCR", "IN", 21.935),
    ("Mexico City", "MX", 20.142),
    ("New York", "US", 20.104),
    ("Sao Paulo", "BR", 19.649),
]
for city in sorted(metro_data, key=itemgetter(1)):
    print(city)

In [ ]:
from functools import partial

# partial freezes some arguments
triple = partial(mul, 3)
print(f"triple(7) = {triple(7)}")
print(f"triple(14) = {triple(14)}")

# Practical: create a specialized tag function
tag_p = partial(tag, "p", class_="highlight")
print(tag_p("Important message"))

---
## Try It Yourself

### Exercise 1: Higher-Order Sort
Write a function `sort_by_vowel_count(words)` that sorts a list of words by the number of vowels they contain (ascending). Use `sorted()` with a `key` function.

In [ ]:
# Exercise 1
def count_vowels(word):
    return sum(1 for c in word.lower() if c in "aeiou")

def sort_by_vowel_count(words):
    return sorted(words, key=count_vowels)

words = ["elephant", "cat", "bee", "automobile", "sky"]
result = sort_by_vowel_count(words)
print(f"By vowel count: {result}")
print(f"Vowel counts:   {[count_vowels(w) for w in result]}")

### Exercise 2: Callable Counter
Create a `Counter` class with `__call__` that counts how many times it has been called and returns the count each time.

In [ ]:
# Exercise 2
class Counter:
    def __init__(self):
        self.count = 0

    def __call__(self):
        self.count += 1
        return self.count

c = Counter()
print(c(), c(), c())  # 1 2 3
print(f"callable? {callable(c)}")

### Exercise 3: Replacing Lambda with operator
Rewrite the following using `operator` module functions instead of `lambda`.

In [ ]:
from operator import itemgetter, attrgetter, add
from functools import reduce
from collections import namedtuple

# itemgetter replaces lambda p: p[2]
people = [("Alice", 30, 85), ("Bob", 25, 92), ("Carol", 28, 78)]
print("Sorted by score:", sorted(people, key=itemgetter(2)))

# operator.add replaces lambda a, b: a + b
numbers = [1, 2, 3, 4, 5]
print("Sum via reduce:", reduce(add, numbers))

# attrgetter replaces lambda s: s.gpa
Student = namedtuple("Student", "name gpa")
students = [Student("Alice", 3.9), Student("Bob", 3.5), Student("Carol", 3.7)]
print("By GPA:", sorted(students, key=attrgetter("gpa")))

---
## Key Takeaways

1. **Functions are objects** -- you can assign, store, pass, and return them just like any other value.

2. **Higher-order functions** (`sorted`, `map`, `filter`, `reduce`) accept functions as arguments. Prefer **list comprehensions** over `map`/`filter` for readability.

3. **lambda** is just sugar for a one-expression `def`. If it is hard to read, refactor to a named function.

4. **`callable()`** is the definitive test for whether an object can be called with `()`.

5. **`__call__`** makes instances callable -- ideal for stateful function-like objects.

6. Use **`/`** (positional-only) and **`*`** (keyword-only) to design clean, unambiguous function signatures.

7. **`operator.itemgetter`**, **`attrgetter`**, **`mul`**, and **`functools.partial`** eliminate trivial lambdas and simplify functional-style code.

---
## See Also

- [[first-class-functions]] -- Functions as first-class objects in depth
- [[higher-order-functions]] -- sorted, map, filter, reduce patterns
- [[replacing-map-filter-reduce]] -- Modern Pythonic alternatives
- [[anonymous-functions-lambda]] -- When to use (and avoid) lambda
- [[callable-types]] -- The nine callable types
- [[user-defined-callable-types]] -- Building callable instances with __call__
- [[parameter-handling]] -- Positional-only and keyword-only parameters
- [[operator-module-and-functools-partial]] -- Functional programming utilities

**Next:** Chapter 8 covers type hints in function signatures.